# ScoreIA — Architecture Snapshot v5.5

Modo: Continuidade Privada Controlada
Status: Fase 5.5 em consolidação

---

# 1. Objetivo deste Documento

Este documento congela o estado estrutural oficial do ScoreIA na Fase 5.5.

Ele define:

* Arquitetura vigente
* Responsabilidades dos módulos
* Pipeline oficial de ML
* Garantias estatísticas
* Decisões arquiteturais imutáveis
* Pendências atuais
* Direção imediata

Este snapshot deve ser tratado como referência canônica para qualquer evolução futura.

---

# 2. Estrutura Atual do Projeto

```
ScoreIA/
│
├── docs/
│   ├── PROJECT_IDENTITY.ipynb
│   ├── ARCHITECTURE_SNAPSHOT_v5_5.ipynb
│   ├── ML_PIPELINE_SPEC.ipynb
│   ├── CODEX_CONTEXT_RULES.ipynb
│   └── ROADMAP_ATUAL.ipynb
│
├── core/
│   ├── agent.py
│   ├── memory.py
│
├── memory/
│   └── profile.json
│
├── tools/
│   ├── study.py
│   ├── planejamento_parser.py
│   ├── data_pipeline.py
│   ├── metrics.py
│   ├── model_factory.py
│   ├── optimization.py
│   ├──interpretability.py
│   ├── webapp/
│       ├── app.py
│       └── template/index.html
│
└── main.py
```

---

# 3. Arquitetura por Camadas

## 3.1 Camada de Orquestração

Arquivo: `core/agent.py`

Responsável por:

* Interpretar comandos
* Acionar módulos adequados
* Gerenciar execução do pipeline ML
* Integrar modo "auto"
* Registrar memória
* Retornar resultados estruturados

Importante:

O agent não contém lógica estatística profunda.
Ele orquestra, não modela.

---

## 3.2 Camada de Memória

Arquivo: `core/memory.py`

Funções:

* Persistência de histórico
* Registro de interações
* Perfil do usuário
* Estado episódico

Escopo atual:

Memória explícita baseada em JSON.

Memória vetorial é futura (Fase 10).

---

## 3.3 Camada de Engenharia de Dados

Arquivo: `tools/data_pipeline.py`

Responsabilidades:

* Carregamento de dataset
* Split holdout
* Inferência de tipos
* Imputação
* Encoding
* Escalonamento
* Transform consistente treino/teste
* Armazenamento de metadata

### Garantia estrutural

O conjunto de teste nunca participa de:

* Fit de imputação
* Fit de encoding
* Fit de escalonamento
* Cross-validation
* Otimização

Fluxo:

```
Dataset bruto
   ↓
Holdout (ex: 80/20)
   ↓
Treino → fit transform
Teste → transform apenas
```

---

## 3.4 Camada de Construção de Modelos

Arquivo: `tools/model_factory.py`

Contém:

* Instanciação de LightGBM
* Definição estrutural de MLP (PyTorch)

Não contém:

* Cross-validation
* Early stopping complexo
* Loop de otimização

---

## 3.5 Camada de Otimização e Treino

Arquivo: `tools/optimization.py`

Componentes principais:

* objective_factory
* treino interno por fold
* early stopping
* otimizar_modelo_cv

### Estratégia Oficial de Cross-Validation

* Folds criados apenas com dados de treino
* Cada trial executa todos os folds
* Métrica final do trial = média dos folds

Fluxo:

```
Treino (80%)
   ↓
Criação de folds
   ↓
Para cada trial:
    Para cada fold:
        Treina
        Avalia
   Média das métricas
↓
Optuna escolhe melhor
```

---

## 3.6 Modo AUTO

Se `modelo="auto"`:

* Otimiza LightGBM
* Otimiza MLP
* Compara best_score (CV)
* Seleciona o melhor modelo
* Treina final apenas o vencedor
* Avalia no teste

A métrica de seleção é exclusivamente a da validação cruzada.

---

## 3.7 Camada de Métricas

Arquivo: `tools/metrics.py`

Responsável por:

* Métricas de regressão
* Métricas de classificação
* Padronização de retorno

Separação clara entre:

* Métrica de CV
* Métrica de teste

---

## 3.8 Interface Web

Diretório: `tools/webapp/`

Função:

* Permitir disparo do pipeline via interface web
* Definir parâmetros
* Visualizar resultados básicos

Estado atual:

Interface funcional, porém ainda separada estruturalmente da execução CLI.

---

## 3.9 main.py

Função atual:

* Entrada principal via terminal
* Inicializa agente

Pendência estrutural:

Adaptar para permitir:

* Execução do pipeline ML via CLI
* Integração futura com backend único
* Unificação com interface web

Essa revisão faz parte oficial da Fase 5.5.

---

# 4. Pipeline Oficial de Machine Learning

## Estratégia Completa

```
Base original
   ↓
Holdout (ex: 20%)
   ↓
Treino (80%)
   ↓
Cross-validation (5 folds)
   ↓
Otimização (Optuna)
   ↓
Treino final com melhores parâmetros
   ↓
Avaliação no teste
```

Teste é intocável até a etapa final.

---

# 5. Garantias Estatísticas

* Não há data leakage
* Teste nunca participa da otimização
* CV é manual e controlada
* Early stopping real implementado
* Métricas separadas por contexto
* Comparação justa entre modelos

---

# 6. Estado Atual da Fase 5.5

## Concluído

* Padronização de funções
* Split correto
* CV real
* Early stopping
* Modo auto
* Interface web inicial

## Pendências

1. SHAP + Feature Importance
2. Exportação do modelo treinado
3. Relatório técnico automático (PDF)
4. Revisão estrutural do main.py (CLI + Web)

---

# 7. Decisões Arquiteturais Imutáveis

1. Agent não contém lógica de modelagem.
2. optimization.py é responsável por CV e treino.
3. data_pipeline.py é responsável por engenharia de dados.
4. Holdout sempre precede CV.
5. Teste nunca entra na otimização.
6. Modularidade deve ser preservada.
7. Webapp não deve conter lógica estatística.

---

# 8. Direção Imediata

Objetivo: Finalizar Fase 5.5.

Ordem recomendada:

1. Implementar interpretabilidade
2. Implementar exportação
3. Implementar relatório técnico
4. Revisar main.py
5. Testar pipeline completo ponta a ponta

---

# 9. Direção Estratégica Pós 5.5

Curto prazo:

* Backend unificado (FastAPI)
* Agente embutido na plataforma

Médio prazo:

* Persistência de modelos
* Multiusuário

Longo prazo:

* Multimodalidade
* Android
* Memória vetorial

---

# 10. Estado Oficial Congelado

ScoreIA v5.5 é:

* Estatisticamente consistente
* Arquiteturalmente modular
* Preparado para evolução
* Em continuidade privada controlada

Qualquer alteração estrutural deve ser explicitamente documentada em um novo snapshot.



## Atualizacao de Arquitetura - Interpretabilidade

### Novo componente
- `tools/interpretability.py`

### Responsabilidade
- Calcular importancia global por tecnica
- Calcular explicacoes locais/semilocais (SHAP para LightGBM, ALE para MLP)
- Reutilizar threshold de classificacao definido na modelagem
- Exportar artefato JSON de interpretabilidade em `reports/`

### Integracao
- `core/agent.py` chama a feature apos treino e avaliacao
- `tools/data_pipeline.py` agora expoe `feature_names_out` para mapear importancias por nome de variavel
